In [12]:
import re
from pathlib import Path

import pandas as pd

In [13]:
raw_lines = [
    "# Lab 5 marks typed by the tutor: matric | name | lab mark (out of 30)",
    "S101 | Aina | 24",
    "s102 |  ben | 20",
    "S103|Chong|28",
    "S104 | Devi | ",
    "",
    "S105 | Emir | 16",
    "  S106 | FARAH | 14  ",
    "S1O7 | Gopal | 25",
    "S108 | Hana | 62",
    "S109 | Irfan | Absent",
    "S110 | Jun Wei | 19.5",
    "S111 Mei Ling 27",
]

# ①
RAW_FILE = Path("lab5_raw.txt")

# ② A newline between the lines, and one at the very end
text = "\n".join(raw_lines) + "\n"

# ③
RAW_FILE.write_text(text, encoding="utf-8")

# ④
text_back = RAW_FILE.read_text(encoding="utf-8")
lines = text_back.splitlines()

# ⑤
print(len(lines), "lines")
for line in lines:
    print(repr(line))

13 lines
'# Lab 5 marks typed by the tutor: matric | name | lab mark (out of 30)'
'S101 | Aina | 24'
's102 |  ben | 20'
'S103|Chong|28'
'S104 | Devi | '
''
'S105 | Emir | 16'
'  S106 | FARAH | 14  '
'S1O7 | Gopal | 25'
'S108 | Hana | 62'
'S109 | Irfan | Absent'
'S110 | Jun Wei | 19.5'
'S111 Mei Ling 27'


In [14]:
# Check: run this after the cell above
assert RAW_FILE.exists(), "③ write text to RAW_FILE"
assert text.count("\n") == len(raw_lines) and text.endswith("S111 Mei Ling 27\n"), "② join the lines with a newline, then add one at the end"
assert lines == raw_lines, "④ read the file back and split it into lines"
print("✅ Task 1 done")

✅ Task 1 done


In [15]:
# A function, so that Task 5 can reuse it on every line
def clean_fields(line):
    parts = line.split("|")                    # ①
    fields = [part.strip() for part in parts]  # ②
    if len(fields) != 3:                    # ③
        return None
    matric = fields[0].upper()                 # ④
    name = fields[1].title()
    mark_text = fields[2]
    return matric, name, mark_text

# Try it on two lines from Task 1
print(clean_fields(lines[2]))
print(clean_fields(lines[12]))

('S102', 'Ben', '20')
None


In [16]:
# Check: run this after the cell above
assert clean_fields("s102 |  ben | 20") == ("S102", "Ben", "20"), "① ② ④ split at |, strip every piece, then fix the case"
assert clean_fields("  S106 | FARAH | 14  ") == ("S106", "Farah", "14"), "④ the name should be in Title Case"
assert clean_fields("S104 | Devi | ") == ("S104", "Devi", ""), "② an empty mark stays an empty string"
assert clean_fields("S111 Mei Ling 27") is None, "③ a line without exactly 3 pieces gives None"
print("✅ Task 2 done")

✅ Task 2 done


In [17]:
# ① Write the whole pattern as a raw string: r"..."
pattern = r"S\d{3}"

codes = ["S101", "S1O7", "S1011", "XS101", "s101"]

# ② – ④ The frame says "for each code", so write a for loop
for code in codes:
    found = re.search(pattern, code) is not None        # ②
    whole = re.fullmatch(pattern, code) is not None        # ③
    print(f"{code:<6} found={found}  whole={whole}")  # ④

# ⑤ Keep the stricter test as a function for Task 5
def valid_matric(matric):
    return re.fullmatch(pattern, matric) is not None

S101   found=True  whole=True
S1O7   found=False  whole=False
S1011  found=True  whole=False
XS101  found=True  whole=False
s101   found=False  whole=False


In [18]:
# Check: run this after the cell above
assert valid_matric("S101") and valid_matric("S999"), "① S followed by three digits is valid"
assert not valid_matric("S1O7"), "① O (the letter) is not 0 (the digit)"
assert not valid_matric("S1011") and not valid_matric("XS101"), "① the pattern is S, then exactly 3 digits"
assert not valid_matric("s101"), "① the S must be upper case"
assert [re.search(pattern, c) is not None for c in codes] == [True, False, True, True, False], "② search looks for a matching piece anywhere"
print("✅ Task 3 done")

✅ Task 3 done


In [19]:
# ① – ④ A chain of decisions becomes if / elif / else. The first True branch wins.
def mark_problem(mark_text):
    if mark_text == "":               # ①
        return "missing"
    elif mark_text.lower() == "absent":  # ②
        return "absent"
    elif not mark_text.isdigit():      # ③
        return "not a whole number"
    elif int(mark_text) > 30:        # ④
        return "above 30"
    else:
        return "ok"

# Try every kind of mark text
for mark_text in ["24", "", "Absent", "19.5", "62", "30", "0"]:
    print(repr(mark_text), "→", mark_problem(mark_text))

'24' → ok
'' → missing
'Absent' → absent
'19.5' → not a whole number
'62' → above 30
'30' → ok
'0' → ok


In [20]:
# Check: run this after the cell above
assert mark_problem("24") == "ok" and mark_problem("0") == "ok", "an ordinary mark is ok"
assert mark_problem("30") == "ok" and mark_problem("31") == "above 30", "④ 30 itself is allowed; 31 is not"
assert mark_problem("") == "missing", "① an empty string means missing"
assert mark_problem("ABSENT") == "absent", "② compare in lower case, so Absent and ABSENT both match"
assert mark_problem("19.5") == "not a whole number" and mark_problem("-3") == "not a whole number", "③ only digits make a whole number"
print("✅ Task 4 done")

✅ Task 4 done


In [21]:
# ① Start the collections before the loop
accepted = []
problems = []
n_skipped = 0

# ② Open the file and number its lines from 1
with RAW_FILE.open(encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        line = line.strip()

        # ③ Skip blank lines and comments
        if line == "" or line.startswith("#"):
            n_skipped += 1
            continue

        # ④
        fields = clean_fields(line)
        if fields is None:
            problems.append({"line": line_number, "text": line, "reason": "expected 3 fields"})
            continue
        matric, name, mark_text = fields

        # ⑤
        if not valid_matric(matric):
            problems.append({"line": line_number, "text": line, "reason": "matric is not S + 3 digits"})
            continue

        # ⑥ ⑦
        reason = mark_problem(mark_text)
        if reason == "ok":
            accepted.append({"matric": matric, "name": name, "lab": int(mark_text)})
        else:
            problems.append({"line": line_number, "text": line, "reason": reason})

# ⑧
print(f"{len(accepted)} accepted, {len(problems)} problems, {n_skipped} skipped")
for problem in problems:
    print(f"line {problem['line']:>2}: {problem['reason']:<27} {problem['text']!r}")

5 accepted, 6 problems, 2 skipped
line  5: missing                     'S104 | Devi |'
line  9: matric is not S + 3 digits  'S1O7 | Gopal | 25'
line 10: above 30                    'S108 | Hana | 62'
line 11: absent                      'S109 | Irfan | Absent'
line 12: not a whole number          'S110 | Jun Wei | 19.5'
line 13: expected 3 fields           'S111 Mei Ling 27'


In [22]:
# Check: run this after the cell above
assert n_skipped == 2, "③ skip the comment line and the blank line, and count them"
assert [student["matric"] for student in accepted] == ["S101", "S102", "S103", "S105", "S106"], "④ – ⑦ expected 5 accepted students"
assert all(type(student["lab"]) is int for student in accepted), "⑦ store the lab mark as a whole number, not text"
assert [problem["line"] for problem in problems] == [5, 9, 10, 11, 12, 13], "② ⑥ record a problem for every rejected line, numbered from 1"
print("✅ Task 5 done")

✅ Task 5 done


In [23]:
# The tutor's answers, in the order they type them
tutor_answers = ["twenty-two", "220", " 22 "]

def ask(prompt):
    answer = tutor_answers.pop(0)
    print(prompt + answer)
    return answer

# Which record has the missing mark? (_ holds the empty mark text, which we do not need)
missing = [problem for problem in problems if problem["reason"] == "missing"][0]
matric, name, _ = clean_fields(missing["text"])

# ①
mark = None
attempts = 0

# ② Ask again while there is no mark and fewer than 3 tries
while mark is None and attempts < 3:
    attempts += 1                # ③
    raw = ask(f"Lab mark for {name} (try {attempts} of 3): ")
    try:
        value = int(raw.strip())  # ④
    except ValueError:
        print("  Please type digits only")
        continue
    if 0 <= value <= 30:         # ⑤
        mark = value
    else:
        print("  The mark must be from 0 to 30")

# ⑥
if mark is None:
    raise ValueError(f"No valid mark for {name} after 3 tries: ask the lecturer")

# ⑦ New lists, so running this cell twice does not add Devi twice
complete = accepted + [{"matric": matric, "name": name, "lab": mark}]
still_to_check = [problem for problem in problems if problem["reason"] != "missing"]
print(f"{name}: {mark}. {len(complete)} students complete, {len(still_to_check)} lines still to check")

Lab mark for Devi (try 1 of 3): twenty-two
  Please type digits only
Lab mark for Devi (try 2 of 3): 220
  The mark must be from 0 to 30
Lab mark for Devi (try 3 of 3):  22 
Devi: 22. 6 students complete, 5 lines still to check


In [24]:
# Check: run this after the cell above
assert mark == 22 and attempts == 3, "④ ⑤ the first two answers are rejected, and ' 22 ' is accepted on try 3"
assert complete[-1] == {"matric": "S104", "name": "Devi", "lab": 22}, "⑦ add Devi to complete"
assert len(complete) == 6 and [problem["line"] for problem in still_to_check] == [9, 10, 11, 12, 13], "⑦ only the missing-mark problem is solved"
print("✅ Task 6 done")

✅ Task 6 done


In [25]:
# ① exist_ok=True: no error if the folder is already there
OUT_DIR = Path("exercise2b_outputs")
OUT_DIR.mkdir(exist_ok=True)

# ② A list of dictionaries becomes a table: one dictionary per row
lab = pd.DataFrame(complete).sort_values("matric")
to_check = pd.DataFrame(still_to_check)

# ③ Leave out the row labels
lab.to_csv(OUT_DIR / "lab5_clean.csv", index=False, encoding="utf-8")
to_check.to_csv(OUT_DIR / "lab5_to_check.csv", index=False, encoding="utf-8")

# ④ Read the file you actually saved. A matric is a label, so keep it as text
saved = pd.read_csv(OUT_DIR / "lab5_clean.csv", dtype={"matric": "string"})

# ⑤ One question about the whole file, so an ordinary if
if saved.shape != lab.shape or saved["matric"].tolist() != lab["matric"].tolist():
    raise ValueError("The saved file is not the table you meant to save")

# ⑥
print(saved)

  matric   name  lab
0   S101   Aina   24
1   S102    Ben   20
2   S103  Chong   28
3   S104   Devi   22
4   S105   Emir   16
5   S106  Farah   14


In [26]:
# Check: run this after the cell above
header = (OUT_DIR / "lab5_clean.csv").read_text(encoding="utf-8").splitlines()[0]
assert header == "matric,name,lab", "③ the first line should be the header only: leave out the index"
assert saved["matric"].tolist() == ["S101", "S102", "S103", "S104", "S105", "S106"], "② sort by matric"
assert saved["lab"].tolist() == [24, 20, 28, 22, 16, 14], "④ read the clean file back"
assert len(pd.read_csv(OUT_DIR / "lab5_to_check.csv")) == 5, "③ save the 5 lines still to check"
print("✅ Task 7 done")

✅ Task 7 done


In [27]:
# ① Start a list that holds the title line
report_lines = ["Lab 5 marks (out of 30)"]

# ② ③ ④ Loop through each row of saved, compute percentage, determine PASS/FAIL, and format
for student in saved.to_dict("records"):
    lab = student["lab"]
    pct = lab / 30 * 100
    if pct >= 50:
        result = "PASS"
    else:
        result = "FAIL"

    line = f"{student['matric']}  {student['name']:<8}{student['lab']:>3}  {pct:5.1f}%  {result}"
    report_lines.append(line)

# ⑤ Add a footer: how many accepted, and the line numbers to check
accepted_count = len(saved)
check_lines = ", ".join([str(p["line"]) for p in still_to_check])
footer = f"{accepted_count} students accepted. Lines to check in lab5_raw.txt: {check_lines}"
report_lines.append(footer)

# ⑥ Join the lines with newlines; write them to lab5_report.txt
REPORT_FILE = OUT_DIR / "lab5_report.txt"
report_text_content = "\n".join(report_lines) + "\n"
REPORT_FILE.write_text(report_text_content, encoding="utf-8")

# ⑦ Read the report back and show it
report_text = REPORT_FILE.read_text(encoding="utf-8")
print(report_text)

Lab 5 marks (out of 30)
S101  Aina     24   80.0%  PASS
S102  Ben      20   66.7%  PASS
S103  Chong    28   93.3%  PASS
S104  Devi     22   73.3%  PASS
S105  Emir     16   53.3%  PASS
S106  Farah    14   46.7%  FAIL
6 students accepted. Lines to check in lab5_raw.txt: 9, 10, 11, 12, 13



In [28]:
# Check: run this after the cell above
report = report_text.splitlines()
assert REPORT_FILE.exists(), "⑥ write the report to REPORT_FILE"
assert len(report) == 8, "expected a title line, 6 student lines and a footer, one per line"
assert report[1].split() == ["S101", "Aina", "24", "80.0%", "PASS"], "④ expected: S101  Aina  24  80.0%  PASS"
assert report[6].split() == ["S106", "Farah", "14", "46.7%", "FAIL"], "③ Farah is below 50%"
assert report[-1].startswith("6 students accepted") and report[-1].endswith("9, 10, 11, 12, 13"), "⑤ footer: the count, then the line numbers to check"
print("✅ Task 8 done: great work!")

✅ Task 8 done: great work!
